# Meal Combination Scoring Sandbox
_Disclaimer: This code is not final, but just an exploration. The storage and data structures used are simply for the purpose of this notebook._

## Load Data

In [1]:
import json
import pandas as pd

with open('../restaurants.menu_item_variations.json', 'r') as file:
    item_list = json.load(file)
file.close()

## Select all Items for Specific Restaurants: Chick-fil-A, Taco Bell, and Chipotle

In [16]:
chick_fil_a_items = []
tacobell_items = []
chipotle_items = []

for item in item_list:
    if item.get("restaurant_name") == "Chick-fil-A":
        chick_fil_a_items.append(item)

    elif item.get("restaurant_name") == "Taco Bell":
        tacobell_items.append(item)
    
    elif item.get("restaurant_name") == "Chipotle":
        chipotle_items.append(item)

print(f"Number of Chick-fil-A items: {len(chick_fil_a_items)}")
print(f"Number of Taco Bell items: {len(tacobell_items)}")
print(f"Number of Chipotle items: {len(chipotle_items)}")

# making dataframes just to visualize easier
df_cfa = pd.DataFrame(chick_fil_a_items)
df_tb = pd.DataFrame(tacobell_items)
df_chipotle = pd.DataFrame(chipotle_items)

df_cfa.reset_index(inplace=True)
df_tb.reset_index(inplace=True)
df_chipotle.reset_index(inplace=True)

Number of Chick-fil-A items: 177
Number of Taco Bell items: 205
Number of Chipotle items: 99


In [17]:
# extract nutrition information as columns
def extract_nutrition(df):
    nutrition_df = df["nutrition_info"].apply(pd.Series)
    df = pd.concat([df.drop(columns=["nutrition_info"]), nutrition_df], axis=1)
    return df

df_cfa = extract_nutrition(df_cfa)
df_tb = extract_nutrition(df_tb)
df_chipotle = extract_nutrition(df_chipotle)

## Exploring Data

### Some Thoughts After Exploring Data

Issue 1: broader categories like "Kid's Meal" and "Breakfast" in Chick fil A contain items that may be from different categories.
- I'm just going to hard code the differences for this example
    - most items in "Breakfast" category can probably be an "Entree" (with the exception of hash browns)
    - items in the "Kid's Meal" are an entree if they have the word "meal" in the name, "milk" for beverage, and the rest would be sides

Issue 2: when computing combinations in the future, should we try to account for 'realistic' combinations?
- If you order a salad, you would probably want salad dressing to go with it. Blindly assigning meals + sides/add-ons won't always be 'realistic'
- Most likely, you wouldn't want to have salad dressing as an add-on when you ordered a chicken sandwich.

Issue 3: "Build-your-own" restaurants (Chipotle) don't exactly map to the 5 categories
- Possible idea: rather than starting off as an "Entree", each item could instead be labeled as a "Base." A "Base" might require that certain conditions are met before it becomes an "Entree." However, this might be hard to generalize:
    - For example, a Burrito Bowl isn't a bowl yet if it contains only protein.
    - A Burrito is basically still a Taco (tortilla + protein) until you add more toppings.
    - However, a user could possibly be satisfied with a Taco being purely tortilla + protein.

## Assign Meal Categories (Entree, Side, Drink, Dessert, Add-on)
Note: this is hard-coded just to have categories to work with.

### Chick-fil-A


In [18]:
# NOTE: REPLACE WITH LLM CATEOGORIZATION
# add a column for meal categories
df_cfa['meal_category'] = 'temp'

# assign meal categories
# note: hard-coded assignment

cfa_entree_categories = ['Classic Chicken', 'Breakfast', 'Cool Wraps', 'Salads', ]
df_cfa.loc[df_cfa['category'].isin(cfa_entree_categories), 'meal_category'] = 'Entree'

df_cfa.loc[df_cfa['category'] == 'Sides', 'meal_category'] = 'Side'

df_cfa.loc[df_cfa['category'] == 'Beverages', 'meal_category'] = 'Drink'

df_cfa.loc[df_cfa['category'] == 'Desserts', 'meal_category'] = 'Dessert'

addon_categories = ['Salad Dressings', 'Sauces']
df_cfa.loc[df_cfa['category'].isin(addon_categories), 'meal_category'] = 'Add-on'

# assigning some categories manually
df_cfa.loc[90, 'meal_category'] = 'Side' # hash brown

kid_mask = df_cfa['category'] == "Kid's Meal"
df_cfa.loc[kid_mask & df_cfa['menu_item_name'].str.contains('Meal', case=False, na=False),'meal_category'] = 'Entree'
df_cfa.loc[kid_mask & df_cfa['menu_item_name'].str.contains('Milk', case=False, na=False),'meal_category'] = 'Drink'
df_cfa.loc[kid_mask & ~df_cfa['menu_item_name'].str.contains('Meal|Milk', case=False, na=False),'meal_category'] = 'Side'


Sorting the menu items for CFA by category:

In [19]:
cfa_entrees = df_cfa.loc[df_cfa['meal_category'] == 'Entree'].copy()
cfa_sides = df_cfa.loc[df_cfa['meal_category'] == 'Side'].copy()
cfa_drinks = df_cfa.loc[df_cfa['meal_category'] == 'Drink'].copy()
cfa_desserts = df_cfa.loc[df_cfa['meal_category'] == 'Dessert'].copy()
cfa_addons = df_cfa.loc[df_cfa['meal_category'] == 'Add-on'].copy()

print(f"Count of Entrees: {len(cfa_entrees)}")
print(f"Count of Sides: {len(cfa_sides)}")
print(f"Count of Drinks: {len(cfa_drinks)}")
print(f"Count of Desserts: {len(cfa_desserts)}")
print(f"Count of Add-Ons: {len(cfa_addons)}")

Count of Entrees: 76
Count of Sides: 19
Count of Drinks: 43
Count of Desserts: 24
Count of Add-Ons: 15


### Taco Bell

Items of concern:
- Cheesy roll up in 'Specialties': not sufficient to be an entree
- Cinnabon Delights (2, 4, and 12 pc): I will put in "Dessert" for now

In [20]:
# add a column for meal categories
df_tb['meal_category'] = 'temp'

tb_entree_categories = ['Burritos', 'Tacos', 'Breakfast', 'Cantina Chicken','Chalupas', 'Nachos', 'Specialties']
df_tb.loc[df_tb['category'].isin(tb_entree_categories), 'meal_category'] = 'Entree'

df_tb.loc[df_tb['category'] == 'Sides', 'meal_category'] = 'Side'

df_tb.loc[df_tb['category'] == 'Beverages', 'meal_category'] = 'Drink'

df_tb.loc[df_tb['category'] == "Condiments & Sauces", 'meal_category'] = 'Add-on'

# manual assignments
df_tb.loc[91, 'meal_category'] = 'Side'     # hash brown
df_tb.loc[194, 'meal_category'] = 'Side'    # cheesy roll up
df_tb.loc[df_tb['menu_item_name'] == "Cinnabon Delights", 'meal_category'] = 'Dessert'

Sorting the menu items for Taco Bell by category:

In [21]:
tb_entrees = df_tb.loc[df_tb['meal_category'] == 'Entree'].copy()
tb_sides = df_tb.loc[df_tb['meal_category'] == 'Side'].copy()
tb_drinks = df_tb.loc[df_tb['meal_category'] == 'Drink'].copy()
tb_desserts = df_tb.loc[df_tb['meal_category'] == 'Dessert'].copy()
tb_addons = df_tb.loc[df_tb['meal_category'] == 'Add-on'].copy()

print(f"Count of Entrees: {len(tb_entrees)}")
print(f"Count of Sides: {len(tb_sides)}")
print(f"Count of Drinks: {len(tb_drinks)}")
print(f"Count of Desserts: {len(tb_desserts)}")
print(f"Count of Add-Ons: {len(tb_addons)}")

Count of Entrees: 75
Count of Sides: 16
Count of Drinks: 87
Count of Desserts: 3
Count of Add-Ons: 24


### Chipotle: TBD

## Meal Combination Storage and Computation Ideas

1. Knapsack Approach
    - 0/1 Knapsack problem: 
            
        _"Given two arrays, `profit[]` and `weight[]`, where each element represents the profit and weight of an item respectively, also given an integer `W` representing the maximum capacity of the knapsack (the total weight it can hold)._
        
        _Put the items into the knapsack such that the sum of profits associated with them is the maximum possible, without exceeding the capacity W."_
        
        (https://www.geeksforgeeks.org/dsa/0-1-knapsack-problem-dp-10/)
    - Concern:
        - Knapsack tries to maximize a single value from a pool of items and returns a best solution. We want to hit targets (protein, calories, price) without losing structure to meals (maintain rules with categories).
        - We want to have multiple top picks and rank results. We would have to adjust knapsack to give best k results.
        - Overall, knapsack on its own may not be sufficient for meal building. However, it may have a use when trying to maximize leftover calories.

2. Meal Precomputation
    - Storing every single meal combination is inefficient
    - Recomputing every single meal after a user request is also inefficient
    - Another note: allowing for multiple entrees (maybe max 2-3) can allow for goals to be hit quicker. 
    - Hybrid architecture idea:
        1. Precompute entree combinations. Store UID, item IDs, protein total, calorie total, price total
        2. Filter dynamically after user interaction. Use boolean masking/vectorization (quick) to filter for acceptable protein, price, and calorie ranges.
        3. Add on optional sides or drinks (add-ons can come in later).
        4. Apply scoring algorithm.
    - Expansion: add some sort of caching mechanism for common presets

    - Edits to this approach: _all_ entree combinations may also explode as system grows. We can possibly filter down "unreasonable" combinations to reduce storage.

### Entree Precomputation
It is useful to adjust the storage of the entrees before calculating their combinations.

In [33]:
import itertools

# note: since item_ids and menu_item_ids have duplicates for variations of an item, we are using the index as a placeholder for an ID
cfa_entrees_dict = cfa_entrees[["index", "menu_item_name", "price", "calories", "protein", "serving_size"]].to_dict("records")

# these are based on the filters within Eatery
MAX_PRICE = 100.0
MAX_CALORIES = 3000   
MAX_PROTEIN = 150

# max_count sets how many entrees you want to combine
def calculate_entree_combos(entrees, max_count=2):
    entree_combos = []
    combo_id = 1    # change id numbering later, maybe

    for k in range(1, max_count+1):
        for combo in itertools.combinations_with_replacement(entrees, k):
            # calculate totals
            total_price = sum(item['price'] for item in combo)
            total_calories = sum(item['calories'] for item in combo)
            total_protein = sum(item['protein'] for item in combo)


            if total_price > MAX_PRICE:
                continue
            if total_calories > MAX_CALORIES:
                continue
            if total_price > MAX_PRICE:
                continue

            entree_combos.append({
                "combo_id": combo_id,
                "entree_ids": tuple(item['index'] for item in combo),
                "entree_names": tuple(item['menu_item_name'] for item in combo),
                "n_entrees": k,
                "price": total_price,
                "calories": total_calories,
                "protein": total_protein,
                "serving_size_per_item": tuple(item['serving_size'] for item in combo)
            })

            combo_id += 1
    
    return pd.DataFrame(entree_combos)

In [34]:
def display_entree_combo(row_entry):
    print(f"Combo ID is {row_entry['combo_id']}")
    print(f"There are {row_entry['n_entrees']} entrees in this combo: {row_entry['entree_names']}")
    print(f"Total price is: {row_entry['price']}")
    print(f"Total calories are: {row_entry['calories']}")
    print(f"Total protein is: {row_entry['protein']}\n")

Testing the computation on Chick-fil-A:

In [35]:
import time
start_time = time.time()
cfa_entree_combos = calculate_entree_combos(cfa_entrees_dict, 2)
end_time = time.time()
print(end_time - start_time)
display(len(cfa_entrees_dict))

0.02025127410888672


76

In [36]:
display(cfa_entree_combos)

,combo_id,entree_ids,entree_names,n_entrees,price,calories,protein,serving_size_per_item
0,1,"(0,)","(Chick-n-Strips,)",1,4.29,200,19,"(2 strips,)"
1,2,"(1,)","(Chicken Nuggets,)",1,3.49,130,13,"(4 nuggets,)"
2,3,"(2,)","(Chicken Nuggets,)",1,4.99,250,27,"(8 nuggets,)"
3,4,"(3,)","(Chicken Nuggets,)",1,14.99,970,103,"(30 nuggets,)"
4,5,"(4,)","(Chick-n-Strips,)",1,6.99,410,39,"(4 strips,)"
...,...,...,...,...,...,...,...,...
2997,2998,"(170, 171)","(Spicy Southwest Salad, Spicy Southwest Salad)",2,21.98,1080,80,"(1 salad, 1 salad)"
2998,2999,"(170, 172)","(Spicy Southwest Salad, Spicy Southwest Salad)",2,21.98,1250,84,"(1 salad, 1 salad)"
2999,3000,"(171, 171)","(Spicy Southwest Salad, Spicy Southwest Salad)",2,21.98,960,78,"(1 salad, 1 salad)"
3000,3001,"(171, 172)","(Spicy Southwest Salad, Spicy Southwest Salad)",2,21.98,1130,82,"(1 salad, 1 salad)"


Display a few combinations from Chick-fil-A:

In [37]:
display_entree_combo(cfa_entree_combos.loc[0])
display_entree_combo(cfa_entree_combos.loc[1000])
display_entree_combo(cfa_entree_combos.loc[2900])

Combo ID is 1
There are 1 entrees in this combo: ('Chick-n-Strips',)
Total price is: 4.29
Total calories are: 200
Total protein is: 19

Combo ID is 1001
There are 2 entrees in this combo: ('Smokehouse BBQ Bacon Sandwich', 'Hash Brown Scramble Burrito')
Total price is: 11.98
Total calories are: 1300
Total protein is: 71

Combo ID is 2901
There are 2 entrees in this combo: ('Market Salad', 'Spicy Southwest Salad')
Total price is: 20.98
Total calories are: 770
Total protein is: 62



Some notes:
- This is still not accounting for variations of an item. For example, this blind combination will combine a 4pc nugget with a 8pc nugget.

### Adding Side or Drink

Exploring adding on sides or drinks to entrees, entree combos, or other meal combinations. Taking the hybrid approach, this will occur after a user has interacted with the system. 

General flow for adding on sides/drinks:
1. Filter based on user constraints.
    - Calculate totals for price, calories, and protein from the input
    - Calculate remaining room for each target category based on the input 
         - e.g. `remaining_price = target_price - input_price`
    - Only take candidate items that are less than or equal to the remaining room plus a tolerance 
        - e.g. `item_price <= (remaining_price + price_tolerance)`
2. Build combinations
    - For each item in the candidate results, combine it with the input and calculate totals for each constraint category.

Later, the resulting combinations can be checked to see if they fall within the acceptable tolerance range for each constraint category. This can be used during scoring and ranking.

### Building a "Full Meal"

To simplify things, we'll define a "full meal" as a combination containing Entree(s), Side, and Drink. We are also accounting for the fact that a user may want to build a full meal for a seed item that is not necessarily an Entree. Whether the input is an Entree, Side, or Drink, it should be expanded until it contains items from each category.

Flow:
1. Evaluate current state of input and check for missing categories
2. Expand each category:
    - Filter candidates
    - Generate expanded meals -> partial meal
        - Early pruning: score in between and remove unreasonable choices, keep top k results. This helps prevent combinatorial explosion.
    - Repeat for missing categories

Final outputs are ranked again after the full meal is built. In the case that a user starts with a Entree seed, add an additional layer to retrieve all entree combos that contain that item.

In the future, this will need to be expanded to additional categories (e.g. add-ons). These ideas are also for the assumption of restaurants that do not follow the "build-your-own" style.


In [ ]:
import math
import numpy as np
import copy

# represent user-chosen constraints
USER_PRICE = 14.00
USER_CAL = 800
USER_PROTEIN = 20

# represent tolerance values
PRICE_TOL = round(USER_PRICE * 0.20, 2)
CAL_TOL = math.ceil(USER_CAL * 0.10)
PROTEIN_TOL = math.ceil(USER_PROTEIN * 0.30)

# represent weights (will be dependent on the user)
W_PRICE = 0.80
W_CAL = 0.10
W_PROTEIN = 0.10

# get all entree combos that contain a specific entree seed
def get_entree_combos(seed_id, df_entree_combos, is_entree):
    if is_entree:
        mask = df_entree_combos['entree_ids'].apply(lambda ids: seed_id in ids)
        return df_entree_combos[mask]
    else:
        return df_entree_combos

def get_cand_items(seed_id, category, 
                   df_entree_combos=None, df_sides=None, df_drinks=None, df_desserts=None, df_addons=None, 
                   is_entree=False):
    
    if category == 'Entree':
        return get_entree_combos(seed_id, df_entree_combos, is_entree)
    elif category == 'Side':
        return df_sides
    elif category == 'Drink':
        return df_drinks
    elif category == "Dessert":
        return df_desserts
    elif category == "Add-on":
        return df_addons
    
# def intermediate_prune(meals, k=50):
#     scores = []
#     for meal in meals:
#         price_efficiency = meal['total_price'] / USER_PRICE
#         cal_dev = abs(meal['total_cal'] - USER_CAL) / USER_CAL
#         protein_dev = abs(meal['total_protein'] - USER_PROTEIN) / USER_PROTEIN
#         golden_ratio = (meal['total_protein'] * 10) / max(meal['total_cal'], 1)
#         print(price_efficiency, cal_dev, protein_dev, golden_ratio)

#         # lower score is better
#         score = -W_PRICE * price_efficiency + W_CAL * cal_dev + W_PROTEIN * (protein_dev - golden_ratio)

#         scores.append(score)
    
#     scores = np.array(scores)
#     sorted_indices = scores.argsort()
#     print(sorted_indices)
#     top_k_indices = sorted_indices[:k]
#     return [meals[i] for i in top_k_indices]

def build_meal(seed_id, required_categories, df_restaurant, 
               df_entree_combos=None, df_sides=None, df_drinks=None, df_desserts=None, df_addons=None,
               build_full=True):
    
    seed_item = df_restaurant.loc[seed_id].to_dict()

    current_meals = []

    is_entree = (seed_item.get('meal_category') == 'Entree')

     # if the item is an entree, get all entree combos
    if is_entree and build_full:
        candidate_entrees = get_entree_combos(seed_id=seed_id, df_entree_combos=df_entree_combos, is_entree=is_entree)
        for _, combo in candidate_entrees.iterrows():
            meal_state = {
                'item_ids': list(combo['entree_ids']),
                'total_price': combo['price'],
                'total_cal': combo['calories'],
                'total_protein': combo['protein'],
                'filled_categories': {'Entree'}
            }
            current_meals.append(meal_state)
    else:
        meal_state = {
            'item_ids': [seed_id],
            'total_price': seed_item.get('price'),
            'total_cal': seed_item.get('calories'),
            'total_protein': seed_item.get('protein'),
            'filled_categories': {seed_item.get('meal_category')}
        }
        current_meals.append(meal_state)

    missing_categories = required_categories - current_meals[0].get('filled_categories')

    for category in missing_categories:
        new_meals = []
        
        for meal in current_meals:
            cand_items = get_cand_items(seed_id=seed_id, category=category, 
                                        df_entree_combos=df_entree_combos, 
                                        df_sides=df_sides, 
                                        df_drinks=df_drinks,
                                        df_desserts=df_desserts,
                                        df_addons=df_addons,
                                        is_entree=is_entree)

            cand_items = cand_items[
                (meal['total_price'] + cand_items['price'] <= USER_PRICE + PRICE_TOL) &
                (meal['total_cal'] + cand_items['calories'] <= USER_CAL + CAL_TOL)
            ]

            for _, item in cand_items.iterrows():
                new_meal = copy.deepcopy(meal)
                
                if category == 'Entree':
                    new_meal['item_ids'].extend(item['entree_ids'])
                else:
                    new_meal['item_ids'].append(item['index'])
                
                new_meal['total_price'] += item['price']
                new_meal['total_cal'] += item['calories']
                new_meal['total_protein'] += item['protein']
                new_meal['filled_categories'].add(category)

                new_meal['total_price'] = round(new_meal['total_price'], 2)
                new_meals.append(new_meal)

        current_meals = intermediate_prune(new_meals)
    
    return current_meals

# some functions for visualization
def get_item_name(df_restaurant, id):
    return df_restaurant.loc[id, 'menu_item_name']

def get_item_price(df_restaurant, id):
    return df_restaurant.loc[id, 'price']

def get_item_calories(df_restaurant, id):
    return df_restaurant.loc[id, 'calories']

def get_item_protein(df_restaurant, id):
    return df_restaurant.loc[id, 'protein']

def display_meal(df_restaurant, meal):
    print(f"Items in this meal:")
    for id in meal['item_ids']:
        print(f"\t{get_item_name(df_restaurant, id)}, ${get_item_price(df_restaurant, id):.2f}, {get_item_calories(df_restaurant, id)}, {get_item_protein(df_restaurant, id)}")
    print(f"Total price: ${meal['total_price']}")
    print(f"Total calories: {meal['total_cal']}")
    print(f"Total_protein: {meal['total_protein']}")
    print(f"Golden ratio: {meal['total_protein'] * 10 / meal['total_cal']:.2f}%\n")

def results_to_df(results, df_restaurant):
    rows = []
    for meal in results:
        row = {
            'items': ', '.join([df_restaurant.loc[id]['menu_item_name'] for id in meal['item_ids']]),
            'total_price': meal['total_price'],
            'total_cal': meal['total_cal'],
            'total_protein': meal['total_protein'],
            'golden_ratio': round((meal['total_protein'] * 10) / max(meal['total_cal'], 1), 4)
        }
        rows.append(row)
    return pd.DataFrame(rows)

In [68]:
def intermediate_prune(meals, k=50):
    scores = []
    print(len(meals))
    for meal in meals:
        # cheaper is always better
        price_efficiency = meal['total_price'] / USER_PRICE
        # calories should always be close to the user's target
        cal_dev = abs(meal['total_cal'] - USER_CAL) / USER_CAL
        # protein should always meet or exceed the user's target
        protein_score = 1 / (meal['total_protein'] / USER_PROTEIN)
        golden_ratio = (meal['total_protein'] * 10) / max(meal['total_cal'], 1)

        # lower score is better
        score = W_PRICE * price_efficiency + W_CAL * cal_dev + W_PROTEIN * (protein_score - golden_ratio)
        scores.append(score)
    
    scores = np.array(scores)
    sorted_indices = scores.argsort()
    top_k_indices = sorted_indices[:k]
    return [meals[i] for i in top_k_indices]

In [ ]:
# categories needed for a full meal
required_categories = {'Entree', 'Side', 'Drink'}

# Chick-fil-A Entree
print(f"Testing on Chick-fil-A item {get_item_name(df_cfa, 4)}\n")

USER_PRICE = 14.00
USER_CAL = 800
USER_PROTEIN = 20

weight_tests = [
    (1, 0, 0),
    (.2, .8, .0),
    (0, 0, 1),
]
pd.set_option('display.max_colwidth', None)
for W_PRICE, W_CAL, W_PROTEIN in weight_tests:
    print(f"Testing with weights - Price: {W_PRICE}, Calories: {W_CAL}, Protein: {W_PROTEIN}")
    results_cfa = build_meal(4, required_categories, df_cfa, cfa_entree_combos, cfa_sides, cfa_drinks, cfa_desserts, cfa_addons)
    display(results_to_df(results_cfa[:3], df_cfa))

Testing on Chick-fil-A item Chick-n-Strips

Testing with weights - Price: 0.2, Calories: 0.8, Protein: 0.0
324
551


,items,total_price,total_cal,total_protein,golden_ratio
0,"Chick-n-Strips, Waffle Fries, Sweet Tea",11.97,800,43,0.5375
1,"Chick-n-Strips, Kids Waffle Potato Fries, Sweet Tea",12.17,800,43,0.5375
2,"Chick-n-Strips, Waffle Fries, Milk",11.57,820,50,0.6098


Issue: the user-set constraints were too limiting for all categories (not enough room). The solver will only create meals that contain all five categories, so it is currently not possible for it to create a meal that is within the tolerance ranges.

Notes: 
- Protein was too limiting for a hard constraint. Users may also want to treat protein targets as minimums rather than maximums.
- Results are still not ranked overall for user constraints and preferences. Results may still seem slightly 'unreasonable' or less than ideal.
- Some results may only make sense in a certain context. Specifically, meals with breakfast items would only be ordered during the set breakfast menu times.
- Combinations are still blind. A user might not want items from the kid's menu or random salad dressings.
- **Need to account for the edge case where constraints are too limiting.**

## Scoring and Ranking

The following approach will treat the values as so:
- Price: budget ceiling
    - If price $\le$ target → no penalty
    - If price $\gt$ target → penalize excess
    - Add preference for cheaper meals
- Calories: maximum constraint
    - If calories $\le$ target → no penalty
    - If calories $\gt$ target → penalize excess
- Protein: minimum constraint (exceeding target will not be penalized)
    - If protein $\ge$ target → no penalty
    - If protein $\lt$ target → penalize deficit

Each will be treated as a soft constraint to maintain enough viable meals but will penalize strongly.

To speed up computation: instead of iterating over each candidate meal, we can instead convert them into a dataframe and perform vectorized operations.

In [75]:
USER_PRICE = 14.0
USER_CAL = 800
USER_PROTEIN = 20

W_PROTEIN = 0.5
W_CAL = 0.35
W_PRICE = 0.1
W_CHEAP = 0.05

In [78]:
def score_and_rank_meals(df_candidate_meals):
    df_candidate_meals['protein_deficit'] = (
        (USER_PROTEIN - df_candidate_meals['total_protein'])
        .clip(lower=0)
        / USER_PROTEIN
    )

    df_candidate_meals['cal_excess'] = (
        (df_candidate_meals['total_cal'] - USER_CAL)
        .clip(lower=0)
        / USER_CAL
    )

    df_candidate_meals['price_excess'] = (
        (df_candidate_meals['total_price'] - USER_PRICE)
        .clip(lower=0)
        / USER_PRICE
    )

    # add a bonus for cheaper meals
    df_candidate_meals['cheap_bonus'] = (
        (USER_PRICE - df_candidate_meals['total_price'])
        .clip(lower=0)
        / USER_PRICE
    )

    df_candidate_meals['golden_ratio'] = (
        (df_candidate_meals['total_protein'] * 10) /
        df_candidate_meals['total_cal'].clip(lower=1)
    )

    # lower score is better
    df_candidate_meals['score'] = (
        W_PROTEIN * df_candidate_meals['protein_deficit'] +
        W_CAL * df_candidate_meals['cal_excess'] +
        W_PRICE * df_candidate_meals['price_excess'] -
        W_CHEAP * df_candidate_meals['cheap_bonus'] -
        W_PROTEIN * df_candidate_meals['golden_ratio']
    )

    top_meals = df_candidate_meals.sort_values('score')

    return top_meals

In [80]:
# test on CFA meals (Entree, Side, Drink)
df_cfa_meals = pd.DataFrame(results_cfa)

df_ranked_cfa = score_and_rank_meals(df_cfa_meals)

display_meal(df_cfa, results_cfa[0])
display_meal(df_cfa, results_cfa[1])
display_meal(df_cfa, results_cfa[2])
display_meal(df_cfa, results_cfa[3])
display_meal(df_cfa, results_cfa[4])
df_ranked_cfa.head()

Items in this meal:
	Chick-n-Strips, $6.99, 410, 39
	Waffle Fries, $2.79, 320, 4
	Sweet Tea, $2.19, 70, 0
Total price: $11.97
Total calories: 800
Total_protein: 43
Golden ratio: 0.54%

Items in this meal:
	Chick-n-Strips, $6.99, 410, 39
	Kids Waffle Potato Fries, $2.99, 320, 4
	Sweet Tea, $2.19, 70, 0
Total price: $12.17
Total calories: 800
Total_protein: 43
Golden ratio: 0.54%

Items in this meal:
	Chick-n-Strips, $6.99, 410, 39
	Waffle Fries, $2.79, 320, 4
	Milk, $1.79, 90, 7
Total price: $11.57
Total calories: 820
Total_protein: 50
Golden ratio: 0.61%

Items in this meal:
	Chick-n-Strips, $6.99, 410, 39
	Kids Waffle Potato Fries, $2.99, 320, 4
	Milk, $1.79, 90, 7
Total price: $11.77
Total calories: 820
Total_protein: 50
Golden ratio: 0.61%

Items in this meal:
	Chick-n-Strips, $6.99, 410, 39
	Waffle Fries, $2.79, 320, 4
	Diet Lemonade, $3.49, 80, 0
Total price: $13.27
Total calories: 810
Total_protein: 43
Golden ratio: 0.53%



,item_ids,total_price,total_cal,total_protein,filled_categories,protein_deficit,cal_excess,price_excess,cheap_bonus,golden_ratio,score
45,"[4, 6, 173, 24]",14.26,780,61,"{Side, Drink, Entree}",0.0,0.00000,0.018571,0.0,0.782051,-0.389168
34,"[4, 6, 173, 19]",14.76,810,61,"{Side, Drink, Entree}",0.0,0.01250,0.054286,0.0,0.753086,-0.366740
29,"[4, 6, 173, 40]",14.26,815,61,"{Side, Drink, Entree}",0.0,0.01875,0.018571,0.0,0.748466,-0.365813
48,"[4, 60, 113, 40]",14.96,790,55,"{Side, Drink, Entree}",0.0,0.00000,0.068571,0.0,0.696203,-0.341244
27,"[4, 12, 149, 24]",15.26,800,56,"{Side, Drink, Entree}",0.0,0.00000,0.090000,0.0,0.700000,-0.341000


## Implementation Ideas/Notes

- Entree Combinations
    - Don't need to store in DB, extra storage
    - Already fast enough if base item is selected

- 

Flow:
- User selects an item
- While user is looking at item / typing their request
    - Query DB for that restaurant's menu items
    - Make entree combos based on selected item
        - If not entree, # of combinations = n + (n+1) choose 2
        - If entree, # of combinations = n + 2
- User sends request, either selecting a template or typing request
    - NLP for request, returns a JSON for what templates are required to fulfill request
    - Still have to figure out specific item requests, like I want fries
- Send result meals in JSON format to frontend

